# 03 — Default Model: Logistic Regression

Spec §5.2. Data source: `panel_logistic_2021_2025.parquet` (2021–2025 originations).

Pipeline:
1. Load panel + FRED macro
2. Build loan-level feature matrix
3. Temporal train/valid/test split
4. Fit LogisticRegression(class_weight='balanced', solver='saga')
5. Calibrate with Platt scaling on validation set
6. Evaluate: ROC curve, PR curve, confusion matrix
7. SHAP feature importance
8. Serialize to artifacts/models/logistic_model.pkl

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from dotenv import load_dotenv
from sklearn.metrics import (
    PrecisionRecallDisplay,
    RocCurveDisplay,
    confusion_matrix,
    precision_recall_curve,
)

load_dotenv()
sys.path.insert(0, str(Path('..') / 'src'))
import default_model as dm
import feature_engineering as fe

PROCESSED = Path('..') / 'data' / 'processed'
FRED_DIR  = Path('..') / 'data' / 'fred'
FIGURES   = Path('..') / 'artifacts' / 'figures'
MODELS    = Path('..') / 'artifacts' / 'models'
FRED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## 1. Load panel and FRED macro

In [ ]:
panel = pd.read_parquet(PROCESSED / 'panel_logistic_2021_2025.parquet')
print(f'Panel shape: {panel.shape}')

macro = fe.fetch_macro_quarterly(
    start='2000-01-01', end='2025-12-31',
    cache_path=FRED_DIR / 'macro_quarterly.csv',
)
print(f'Macro shape: {macro.shape}, index range: {macro.index[0]} – {macro.index[-1]}')
macro.tail()

## 2. Build loan-level feature matrix

In [ ]:
loan_df = fe.build_logistic_features(panel, macro)
print(f'Loan-level dataset: {loan_df.shape}')
print(f'Default rate: {loan_df.default.mean():.4%}')
print(f'Feature columns present: {all(c in loan_df.columns for c in dm.FEATURE_COLS)}')
loan_df[dm.FEATURE_COLS].describe().round(3)

## 3. Temporal train / valid / test split (spec §5.2)

In [ ]:
train_df, valid_df, test_df = dm.temporal_split(
    loan_df, train_cutoff='2024-01-01', valid_cutoff='2024-07-01'
)
for name, split in [('Train', train_df), ('Valid', valid_df), ('Test', test_df)]:
    print(f'{name:5s}: {len(split):>8,} loans | default rate {split.default.mean():.4%}')

## 4–5. Fit logistic regression + Platt calibration

In [ ]:
model, scaler, val_metrics = dm.train_logistic(train_df, valid_df)
print('Validation metrics:')
for key, val in val_metrics.items():
    print(f'  {key}: {val:.4f}' if isinstance(val, float) else f'  {key}: {val:,}')

## 6. Evaluate on test set — ROC, PR curve, confusion matrix

In [ ]:
test_metrics = dm.evaluate_on_test(model, scaler, test_df)
print('Test metrics (spec §5.2 targets: ROC-AUC ~0.83, default rate ~0.32%):')
for key, val in test_metrics.items():
    print(f'  {key}: {val:.4f}' if isinstance(val, float) else f'  {key}: {val:,}')

In [ ]:
# Get probabilities for plots
x_test_raw, y_test = dm._prepare_xy(test_df, dm.FEATURE_COLS)
x_test_scaled = scaler.transform(x_test_raw)
y_prob_test = model.predict_proba(x_test_scaled)[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

RocCurveDisplay.from_predictions(y_test, y_prob_test, ax=axes[0], name='Calibrated LR')
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title(f'ROC Curve (AUC = {test_metrics["test_roc_auc"]:.3f})')

PrecisionRecallDisplay.from_predictions(y_test, y_prob_test, ax=axes[1], name='Calibrated LR')
axes[1].axhline(y_test.mean(), color='r', linestyle='--', alpha=0.6, label='Random baseline')
axes[1].set_title(f'PR Curve (PR-AUC = {test_metrics["test_pr_auc"]:.4f})')
axes[1].legend()

plt.tight_layout()
plt.savefig(FIGURES / 'logistic_roc_pr_curves.png', bbox_inches='tight')
plt.show()

In [ ]:
# Precision/Recall at multiple thresholds
prec, rec, thresholds = precision_recall_curve(y_test, y_prob_test)
threshold_df = pd.DataFrame({'threshold': thresholds, 'precision': prec[:-1], 'recall': rec[:-1]})

# Find threshold closest to 95th percentile score
p95 = np.percentile(y_prob_test, 95)
closest = (threshold_df['threshold'] - p95).abs().idxmin()
row = threshold_df.loc[closest]
print(f'At 95th pct score threshold ({p95:.4f}):')
print(f'  Precision: {row.precision:.3f} (expect ~2%)')
print(f'  Recall:    {row.recall:.3f} (expect ~31%)')

## 7. SHAP feature importance

In [ ]:
x_train_raw, _ = dm._prepare_xy(train_df, dm.FEATURE_COLS)

# Use a 1000-sample background for speed
rng = np.random.default_rng(42)
bg_idx = rng.choice(len(x_train_raw), size=min(1000, len(x_train_raw)), replace=False)
x_background = x_train_raw[bg_idx]

shap_values, expected_val = dm.compute_shap_values(
    model, scaler, x_background, x_test_raw
)

shap.summary_plot(
    shap_values, x_test_raw,
    feature_names=dm.FEATURE_COLS,
    show=False,
)
plt.title('SHAP Summary — Logistic Regression Default Model')
plt.tight_layout()
plt.savefig(FIGURES / 'logistic_shap_summary.png', bbox_inches='tight')
plt.show()

## 8. Serialize model

In [ ]:
model_path = dm.save_model(model, scaler, out_dir=MODELS)
print(f'Model saved to: {model_path}')

# Verify round-trip load
bundle = dm.load_model(model_path)
y_prob_reload = bundle['model'].predict_proba(x_test_scaled)[:, 1]
assert np.allclose(y_prob_test, y_prob_reload), 'Round-trip probabilities differ!'
print('Round-trip load: probabilities match ✓')
print(f'\nFinal test ROC-AUC: {test_metrics["test_roc_auc"]:.4f}')